# Горизонты NII, дисконтированный NII и процентный риск

Повторяем заданные состояния из `policy_scenarios.ipynb` и один шок короткой ставки $+100$ б.п. Сохраняем исходную $\Delta NII_{1y}^{\rm frozen}$ при неизменном балансе в течение года. Для длинных горизонтов используем **пассивный run-off**: нет новых кредитов или депозитов; начальные $L,F,Y,D,C$ уменьшаются с интенсивностью $\mu$. Ставка под физической мерой $P$ возвращается к среднему. Это отдельная исследовательская конвенция, не NII при выученной политике.

Сначала считаем ожидаемое **недисконтированное** изменение NII в отдельном году $j$:

$$\Delta N_j=\delta K\int_{j-1}^{j}e^{-(\mu+\kappa_P)u}du,\qquad K=L-F+C-\beta D,\quad C=E+D-L.$$

Накопленный эффект до горизонта $H$ равен $\Delta NII_H^{\rm runoff}=\sum_{j=1}^H\Delta N_j$. Произвольный показатель $WNII_5$ здесь **убран**.

Для сопоставления с MtM и $V$ вводим приведённую стоимость NII того же пассивного run-off. Пусть $P_P(r,u)=\mathbb E_r^P[\exp(-\int_0^u r_s ds)]$, $M_P(r,u)=\mathbb E_r^P[r_u\exp(-\int_0^u r_s ds)]=-\partial_uP_P(r,u)$ и $a=Y-s_DD-s_B(-C)^+$. Тогда

$$PVNII_H(r)=\int_0^H e^{-\mu u}\{aP_P(r,u)+KM_P(r,u)\}\,du,\qquad \Delta PVNII_H=PVNII_H(r+\delta)-PVNII_H(r).$$

При шоке меняются **и будущие процентные поступления, и их дисконтирование**. Для CIR-процесса под $P$ величины $P_P$ и $M_P$ считаются аналитически; численно интегрируем только по времени. Для наглядности показаны горизонты 2, 3, 5, 10 и 20 лет. Это приведённая стоимость будущего *NII*, а не EVE: погашения основной суммы не входят в NII, а MtM использует оценочную меру $Q$ вместо $P$.


In [ ]:
from pathlib import Path
import json, sys, os, math
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

RUN_DIR = 'runs/stationary_rate_growth'  # поменяйте на нужный чекпоинт
SHOCK = .01
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT = next((q for q in [Path.cwd(), *Path.cwd().parents]
             if (q/'continuous_time_irrbb/risk_sensitive/stationary_solver.py').exists()), None)
if ROOT is None: raise FileNotFoundError('Запустите Jupyter внутри проекта IRRBB')
sys.path.insert(0,str(ROOT))
from continuous_time_irrbb.risk_sensitive.stationary_solver import StationarySolution
run_dir = Path(RUN_DIR)
if not run_dir.is_absolute():run_dir=ROOT/'continuous_time_irrbb/risk_sensitive'/run_dir
report=json.loads((run_dir/'report.json').read_text())
dtype=torch.float64 if report['dtype']=='float64' else torch.float32
solution=StationarySolution.load(run_dir/'stationary_solution.pt',device=DEVICE,dtype=dtype)
env,p=solution.environment,solution.environment.p
L,D=p.loans,p.deposits
print(f'{run_dir.name}: device={DEVICE}; chi={p.growth_rate_sensitivity:g}; '
      f'Bellman RMSE={report["diagnostics"][-1]["bellman_rmse"]:.4g}; '
      f'converged={report.get("converged_on_validation_cloud")}')
plt.style.use('seaborn-v0_8-whitegrid')


## Одинаковые показательные состояния

Текущий размер книги и депозитов одинаков во всех строках; ставка старых fixed-кредитов и спред старых float-кредитов выведены из сохранённого агрегированного состояния. В крайних сценариях они условно остаются на тех же уровнях, чтобы отдельно показать эффект состава книги.

In [ ]:
r0,F0,Y0,E0=map(float,report['reference_state'])
share0=F0/L
old_float_spread=p.floating_spread_0-p.floating_demand_slope*(1-share0)
old_fixed_coupon=(Y0-old_float_spread*(L-F0))/F0
scenarios=[
 (f'Исходная книга: {share0:.0%} fixed',r0,share0,E0/L,old_fixed_coupon),
 ('Низкая ставка, те же договоры',max(.02,r0-.06),share0,E0/L,old_fixed_coupon),
 ('Высокая ставка, те же договоры',r0+.06,share0,E0/L,old_fixed_coupon),
 ('20% fixed в книге',r0,.20,E0/L,old_fixed_coupon),
 ('80% fixed в книге',r0,.80,E0/L,old_fixed_coupon),
 ('Низкая ставка + дорогой fixed',max(.02,r0-.06),.80,E0/L,old_fixed_coupon+.02),
 ('Пониженный капитал',r0,share0,max(.105,E0/L-.02),old_fixed_coupon),
 ('Вся книга — float (0% fixed)',r0,.0,E0/L,old_fixed_coupon),
 ('Вся книга — fixed (100% fixed)',r0,1.0,E0/L,old_fixed_coupon),
]
labels=[s[0] for s in scenarios]
def states_from_specs(specs):
 rows=[]
 for _,r,share,car,coupon in specs:
  f=L*share
  y=f*coupon+(L-f)*old_float_spread
  rows.append((r,f,y,L*car))
 return torch.tensor(rows,device=DEVICE,dtype=dtype)
states=states_from_specs(scenarios)
print(f'L={L:g}, D={D:g}, старый fixed-купон={old_fixed_coupon:.2%}, '
      f'старый float-спред={old_float_spread:.2%}')


## Изменение NII по годам

Первые пять колонок показывают отдельные годы, а не перекрывающиеся накопленные горизонты. На второй карте годовые изменения **дисконтированного** NII: здесь шок меняет также стоимость уже существующих будущих платежей. Это помогает увидеть, в каком году и по какому каналу возникает отличие от простой суммы NII.


In [ ]:
# Аналитические моменты CIR под физической мерой P.
# M_P(r,u)=-∂_u P_P(r,u) учитывает корреляцию ставки и дисконтного множителя.
def cir_discount_moments(rate, maturity):
    k,theta,sigma=p.kappa_p,p.mean_rate_p,p.rate_vol
    u=maturity
    if sigma==0:
        b=-torch.expm1(-k*u)/k
        price=torch.exp(-theta*u-(rate-theta)*b)
        rate_moment=price*(theta+(rate-theta)*torch.exp(-k*u))
    else:
        g=math.sqrt(k*k+2*sigma*sigma)
        decay=torch.exp(-g*u)
        denominator=g+k+(g-k)*decay
        b=2*(-torch.expm1(-g*u))/denominator
        log_a=(2*k*theta/(sigma*sigma))*(math.log(2*g)+(k-g)*u/2-torch.log(denominator))
        price=torch.exp(log_a-b*rate)
        b_prime=1-k*b-.5*sigma*sigma*b*b
        rate_moment=price*(k*theta*b+rate*b_prime)
    return price,rate_moment

TIME_NODES=24
quad_x,quad_w=np.polynomial.legendre.leggauss(TIME_NODES)
quad_x=torch.as_tensor(quad_x,device=DEVICE,dtype=dtype)
quad_w=torch.as_tensor(quad_w,device=DEVICE,dtype=dtype)
PV_HORIZONS=(2,3,5,10,20)

def discounted_annual_delta(r, spread_flow, exposure, years=20):
    annual=[]
    for year in range(years):
        u=year+(quad_x+1)/2
        def pv_rate(current_rate):
            price,moment=cir_discount_moments(current_rate[:,None],u[None,:])
            flow=torch.exp(-p.amortization*u)[None,:]*(spread_flow[:,None]*price
                                                        +exposure[:,None]*moment)
            return (flow*quad_w[None,:]).sum(-1)/2
        annual.append(pv_rate(r+SHOCK)-pv_rate(r))
    return torch.stack(annual,-1)

def metrics_for_states(x):
    shape=x.shape[:-1]
    rows=x.reshape(-1,4)
    outputs=[]
    with torch.no_grad():
        bump=torch.tensor([SHOCK,0,0,0],device=DEVICE,dtype=dtype)
        decay=p.amortization+p.kappa_p
        year=np.arange(5,dtype=float)
        factors=torch.as_tensor((np.exp(-decay*year)-np.exp(-decay*(year+1)))/decay,
                                device=DEVICE,dtype=dtype)
        for batch in rows.split(256):
            r,F,Y,E=batch.unbind(-1)
            cash=E+D-L
            exposure=L-F+cash-p.deposit_beta*D
            spread_flow=Y-p.deposit_spread*D-p.funding_spread*(-cash).clamp_min(0)
            frozen_1y=SHOCK*exposure
            annual=SHOCK*exposure[:,None]*factors[None,:]
            cumulative=annual.cumsum(-1)
            annual_pv=discounted_annual_delta(r,spread_flow,exposure)
            cumulative_pv=annual_pv.cumsum(-1)
            pv_horizons=cumulative_pv[:,[h-1 for h in PV_HORIZONS]]
            delta_mtm=(Y+p.amortization*F)*(env.annuity_q(r+SHOCK)-env.annuity_q(r))
            delta_v=solution.value(batch+bump)-solution.value(batch)
            outputs.append(torch.cat((frozen_1y[:,None],annual,cumulative[:,[1,2,4]],
                                      pv_horizons,delta_mtm[:,None],delta_v[:,None]),-1).cpu())
    return torch.cat(outputs).reshape(*shape,16).numpy()

result=metrics_for_states(states)
# 0: frozen1; 1:6: nominal annual1..5; 6:9: nominal2/3/5;
# 9:14: discounted2/3/5/10/20; 14: MtM; 15: V.
annual=result[:,1:6]
assert np.allclose(result[:,8],annual.sum(1))
print('Сценарий                              frozen1   NII5    PVNII2   PVNII3   PVNII5   PVNII10  PVNII20    MtM       V')
for name,row in zip(labels,result):
    print(f'{name:38s} {row[0]:+8.3f} {row[8]:+8.3f} '
          f'{row[9]:+9.3f} {row[10]:+9.3f} {row[11]:+9.3f} '
          f'{row[12]:+9.3f} {row[13]:+9.3f} {row[14]:+8.3f} {row[15]:+8.3f}')
fig,axes=plt.subplots(1,2,figsize=(16,5),constrained_layout=True)
y=np.arange(len(labels))
for ax,values,title in ((axes[0],annual,'Недисконтированный ΔNII по годам'),
                        (axes[1],result[:,9:14],'ΔPVNII до горизонта H')):
    scaled=100*values/L
    maximum=max(.01,float(np.abs(scaled).max()))
    image=ax.imshow(scaled,aspect='auto',cmap='RdBu_r',vmin=-maximum,vmax=maximum)
    ax.set(yticks=y,yticklabels=labels,title=title)
    fig.colorbar(image,ax=ax,label='% начальной книги')
axes[0].set(xticks=range(5),xticklabels=range(1,6),xlabel='отдельный год после шока')
axes[1].set(xticks=range(5),xticklabels=PV_HORIZONS,xlabel='горизонт H, годы')
plt.show()


## Сравнение риск-метрик по сценариям

Для NII и дисконтированного NII используются **отдельные цветовые шкалы**, поскольку их масштабы различаются. Справа показаны MtM и $V$. В отличие от прежнего $WNII$, все $PVNII_H$ выражены в единицах приведённой денежной стоимости. Но PV от NII всё ещё пропускает потоки основной суммы и использует физическую меру $P$, поэтому равенства с MtM или $V$ не ожидается.


In [ ]:
blocks=[
 ([0,6,7,8],['NII₁ frozen','NII₂ runoff','NII₃ runoff','NII₅ runoff'],
  'Недисконтированный NII'),
 ([9,10,11,12,13],['PVNII₂','PVNII₃','PVNII₅','PVNII₁₀','PVNII₂₀'],
  'Дисконтированный NII'),
 ([14,15],['MtM','V'],'MtM и V'),
]
fig,axes=plt.subplots(1,3,figsize=(20,7),constrained_layout=True,
                      gridspec_kw={'width_ratios':[4,5,2]})
for ax,(indices,names,title) in zip(axes,blocks):
    values=result[:,indices]
    vmax=max(.01,float(np.abs(values).max()))
    image=ax.imshow(values,aspect='auto',cmap='RdBu_r',vmin=-vmax,vmax=vmax)
    ax.set(yticks=range(len(labels)),yticklabels=labels,
           xticks=range(len(names)),xticklabels=names,title=title)
    ax.tick_params(axis='x',rotation=40)
    for i in range(len(labels)):
        for j in range(len(names)):
            ax.text(j,i,f'{values[i,j]:+.2f}',ha='center',va='center',fontsize=8,
                    color='white' if abs(values[i,j])>vmax*.55 else 'black')
    fig.colorbar(image,ax=ax,shrink=.75)
plt.show()


## Зависимость от состава книги и текущей ставки

Меняем отдельно $F/L$ и $r$ вокруг исходной книги, сохраняя $L,D,E$ и условия старых договоров. На верхних рисунках сравниваются исходный однолетний frozen NII и пассивные накопленные горизонты. На нижних — приведённая стоимость NII за 5 и 20 лет с MtM и $V$; хотя все имеют денежную размерность, охват потоков и меры вероятности остаются разными.


In [ ]:
grid_n=17 if os.environ.get('IRRBB_NOTEBOOK_SMOKE') else 51
share_grid=np.linspace(0,1,grid_n)
rate_grid=np.linspace(.02,.18,grid_n)
curve_sets=[
 ('Доля fixed в существующей книге, %',100*share_grid,
  states_from_specs([('share',r0,float(a),E0/L,old_fixed_coupon) for a in share_grid])),
 ('Текущая ставка r, %',100*rate_grid,
  states_from_specs([('rate',float(r),share0,E0/L,old_fixed_coupon) for r in rate_grid])),
]
fig,axes=plt.subplots(2,2,figsize=(15,9),constrained_layout=True)
for col,(xlabel,xgrid,grid_states) in enumerate(curve_sets):
    z=metrics_for_states(grid_states)
    for idx,label in ((0,'NII₁ frozen'),(6,'NII₂ runoff'),
                      (7,'NII₃ runoff'),(8,'NII₅ runoff')):
        axes[0,col].plot(xgrid,100*z[:,idx]/L,label=label)
    for idx,label in ((11,'PVNII₅'),(13,'PVNII₂₀'),(14,'MtM'),(15,'V')):
        axes[1,col].plot(xgrid,100*z[:,idx]/L,label='Δ'+label)
    for row in range(2):
        ax=axes[row,col]
        ax.axhline(0,color='black',lw=.7)
        ax.set(xlabel=xlabel,ylabel='% начальной книги')
        ax.legend(fontsize=8,ncol=2)
axes[0,0].set_title('Горизонты NII: состав книги')
axes[0,1].set_title('Горизонты NII: ставка')
axes[1,0].set_title('Приведённая стоимость: состав книги')
axes[1,1].set_title('Приведённая стоимость: ставка')
plt.show()


**Интерпретация.** Пассивный NII описывает погашение действующей книги без управления; $PVNII_H$ добавляет дисконтирование ожидаемых потоков под физической мерой $P$. MtM оценивает все платежи действующих активов по $Q$ и принимает депозиты по номиналу; $V$ включает будущую выдачу, выплаты владельцу и риск-чувствительную рекурсию. Сближение $\Delta PVNII_H$ с $\Delta MtM$ или $\Delta V$ при росте $H$ — эмпирический результат для данной калибровки, не тождество. Для $\Delta V$ учитывайте невязку Беллмана и статус сходимости чекпоинта.
